# 🌋 GempaRadar — Spark Analysis
**Mata Kuliah:** Big Data dan Data Lakehouse  
**Anggota Kelompok:** 
**Topik:** Gempa Radar — Monitor Aktivitas Seismik Indonesia

Notebook ini membaca data dari HDFS (`/data/gempa/`), menjalankan 3 analisis wajib,
dan menyimpan hasil ke HDFS serta `dashboard/data/spark_results.json`.

In [1]:
# : Setup SparkSession dengan koneksi ke HDFS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json, os

spark = (
    SparkSession.builder
    .appName("GempaRadar-Analysis")
   
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("✓ SparkSession berhasil dibuat")
print(f"  Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 04:41:27 WARN Utils: Your hostname, LAPTOP-I4SPJSU4, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 04:41:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 04:41:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✓ SparkSession berhasil dibuat
  Spark version: 4.1.1


In [2]:
# : Baca data gempa dari HDFS
import os
os.makedirs("data/hasil", exist_ok=True)

HDFS_API = "data/api/"
HDFS_RSS = "data/rss/"
HDFS_HASIL = "data/hasil/"

# Baca semua file JSON dari folder API
df_api = (
    spark.read
    .option("multiLine", True)
    .json(HDFS_API)
)

# Baca semua file JSON dari folder RSS
df_rss = (
    spark.read
    .option("multiLine", True)
    .json(HDFS_RSS)
)

print(f"✓ Data API: {df_api.count()} event gempa")
print(f"✓ Data RSS: {df_rss.count()} artikel berita")
print("\n--- Schema Data API ---")
df_api.printSchema()
print("\n--- Sample Data API ---")
df_api.show(5, truncate=50)

✓ Data API: 30 event gempa
✓ Data RSS: 419 artikel berita

--- Schema Data API ---
root
 |-- depth_km: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ingested_at: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- magnitude: double (nullable = true)
 |-- place: string (nullable = true)
 |-- source: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tsunami: long (nullable = true)
 |-- url: string (nullable = true)
 |-- wilayah: string (nullable = true)


--- Sample Data API ---
+--------+----------+--------------------+--------+---------+---------+---------------------------------------+------+--------+--------------------+-------+--------------------------------------------------+--------------------------+
|depth_km|  event_id|         ingested_at|latitude|longitude|magnitude|                                  place|source|  status|           tim

## Analisis 1 — Distribusi Magnitudo
Kategorikan setiap event gempa berdasarkan magnitudonya dan hitung distribusi per kategori.

In [3]:
# : Analisis 1 — Distribusi Magnitudo menggunakan DataFrame API

df_mag = df_api.withColumn(
    "kategori_mag",
    F.when(F.col("magnitude") < 3.0, "Mikro (< 3.0)")
     .when((F.col("magnitude") >= 3.0) & (F.col("magnitude") < 4.0), "Minor (3.0–3.9)")
     .when((F.col("magnitude") >= 4.0) & (F.col("magnitude") < 5.0), "Sedang (4.0–4.9)")
     .when(F.col("magnitude") >= 5.0, "Kuat (≥ 5.0)")
     .otherwise("Tidak Diketahui")
)

hasil_mag = (
    df_mag.groupBy("kategori_mag")
    .agg(
        F.count("*").alias("jumlah"),
        F.round(F.avg("magnitude"), 2).alias("rata_rata_mag"),
        F.round(F.max("magnitude"), 2).alias("mag_tertinggi"),
    )
    .orderBy("jumlah", ascending=False)
)

# Hitung persentase
total = df_api.count()
hasil_mag = hasil_mag.withColumn(
    "persentase",
    F.round(F.col("jumlah") / total * 100, 1)
)

print("=== ANALISIS 1: Distribusi Magnitudo Gempa ===")
hasil_mag.show(truncate=False)

# Simpan ke HDFS
hasil_mag.write.mode("overwrite").json(f"{HDFS_HASIL}/distribusi_magnitudo")
print("✓ Hasil Analisis 1 tersimpan ke HDFS")

=== ANALISIS 1: Distribusi Magnitudo Gempa ===
+----------------+------+-------------+-------------+----------+
|kategori_mag    |jumlah|rata_rata_mag|mag_tertinggi|persentase|
+----------------+------+-------------+-------------+----------+
|Sedang (4.0–4.9)|23    |4.54         |4.9          |76.7      |
|Kuat (≥ 5.0)    |7     |5.27         |5.7          |23.3      |
+----------------+------+-------------+-------------+----------+

✓ Hasil Analisis 1 tersimpan ke HDFS


### Interpretasi Analisis 1
Distribusi magnitudo memberikan gambaran tentang tingkat aktivitas seismik Indonesia.
Gempa **Mikro (< 3.0)** dan **Minor (3.0–3.9)** mendominasi frekuensi, yang konsisten dengan
posisi Indonesia di Cincin Api Pasifik — aktivitas seismik rendah terjadi hampir setiap jam.
Gempa **Kuat (≥ 5.0)** yang muncul dalam data perlu mendapat perhatian BPBD karena
berpotensi merusak bangunan di atas skala MMI V.

## Analisis 2 — Wilayah Paling Aktif
Identifikasi 10 wilayah dengan frekuensi gempa tertinggi menggunakan Spark SQL.

In [4]:
# : Analisis 2 — Wilayah Paling Aktif menggunakan Spark SQL

df_mag.createOrReplaceTempView("gempa")

hasil_wilayah = spark.sql("""
    SELECT
        wilayah,
        COUNT(*) AS jumlah_gempa,
        ROUND(AVG(magnitude), 2) AS rata_rata_mag,
        ROUND(MAX(magnitude), 2) AS mag_tertinggi,
        ROUND(MIN(magnitude), 2) AS mag_terendah,
        SUM(CASE WHEN magnitude >= 5.0 THEN 1 ELSE 0 END) AS gempa_kuat
    FROM gempa
    WHERE wilayah IS NOT NULL AND wilayah != 'Unknown'
    GROUP BY wilayah
    ORDER BY jumlah_gempa DESC
    LIMIT 10
""")

print("=== ANALISIS 2: Top 10 Wilayah Paling Aktif Seismik ===")
hasil_wilayah.show(truncate=60)

# Simpan ke HDFS
hasil_wilayah.write.mode("overwrite").json(f"{HDFS_HASIL}/wilayah_aktif")
print("✓ Hasil Analisis 2 tersimpan ke HDFS")

=== ANALISIS 2: Top 10 Wilayah Paling Aktif Seismik ===


+--------------------------+------------+-------------+-------------+------------+----------+
|                   wilayah|jumlah_gempa|rata_rata_mag|mag_tertinggi|mag_terendah|gempa_kuat|
+--------------------------+------------+-------------+-------------+------------+----------+
|        Ternate, Indonesia|           6|         4.67|          5.1|         4.4|         1|
|         Modisi, Indonesia|           5|          4.8|          5.1|         4.4|         2|
|Pante Makasar, Timor Leste|           2|          5.0|          5.5|         4.5|         1|
|     Lospalos, Timor Leste|           2|          4.4|          4.4|         4.4|         0|
|          Ambon, Indonesia|           2|          4.5|          4.5|         4.5|         0|
|         Bitung, Indonesia|           2|         4.55|          4.7|         4.4|         0|
|   Gunungsitoli, Indonesia|           2|          5.0|          5.7|         4.3|         1|
|           Tual, Indonesia|           2|          4.6|     

### Interpretasi Analisis 2
Wilayah dengan frekuensi gempa tertinggi umumnya berada di zona subduksi aktif:
perairan sekitar **Maluku, Sulawesi, Papua, dan Nusa Tenggara** secara historis mendominasi
peta seismisitas Indonesia. BPBD di wilayah-wilayah ini perlu memastikan sistem early-warning
dan rencana evakuasi selalu dalam kondisi siap. Kolom `gempa_kuat` menunjukkan wilayah
mana yang memerlukan perhatian lebih dalam periode monitoring ini.

## Analisis 3 — Distribusi Kedalaman
Analisis apakah gempa lebih banyak terjadi dangkal atau dalam, dan implikasinya.

In [5]:
# : Analisis 3 — Distribusi Kedalaman menggunakan Spark SQL

hasil_kedalaman = spark.sql("""
    SELECT
        CASE
            WHEN depth_km < 70  THEN 'Dangkal (< 70 km)'
            WHEN depth_km < 300 THEN 'Menengah (70–300 km)'
            ELSE                     'Dalam (> 300 km)'
        END AS kategori_kedalaman,
        COUNT(*) AS jumlah,
        ROUND(AVG(depth_km), 1) AS rata_rata_depth_km,
        ROUND(MIN(depth_km), 1) AS min_depth_km,
        ROUND(MAX(depth_km), 1) AS max_depth_km,
        ROUND(AVG(magnitude), 2) AS rata_rata_mag
    FROM gempa
    WHERE depth_km IS NOT NULL
    GROUP BY kategori_kedalaman
    ORDER BY jumlah DESC
""")

print("=== ANALISIS 3: Distribusi Kedalaman Gempa ===")
hasil_kedalaman.show(truncate=False)

# Statistik tambahan
stats_depth = spark.sql("""
    SELECT
        ROUND(AVG(depth_km), 2) AS avg_depth_km,
        ROUND(STDDEV(depth_km), 2) AS std_depth_km,
        ROUND(PERCENTILE_APPROX(depth_km, 0.5), 1) AS median_depth_km
    FROM gempa
    WHERE depth_km IS NOT NULL
""")
print("--- Statistik Kedalaman Keseluruhan ---")
stats_depth.show()

# Simpan ke HDFS
hasil_kedalaman.write.mode("overwrite").json(f"{HDFS_HASIL}/distribusi_kedalaman")
print("✓ Hasil Analisis 3 tersimpan ke HDFS")

=== ANALISIS 3: Distribusi Kedalaman Gempa ===
+--------------------+------+------------------+------------+------------+-------------+
|kategori_kedalaman  |jumlah|rata_rata_depth_km|min_depth_km|max_depth_km|rata_rata_mag|
+--------------------+------+------------------+------------+------------+-------------+
|Dangkal (< 70 km)   |16    |29.2              |10.0        |40.8        |4.76         |
|Menengah (70–300 km)|9     |132.1             |73.7        |205.3       |4.73         |
|Dalam (> 300 km)    |5     |417.4             |305.4       |500.3       |4.52         |
+--------------------+------+------------------+------------+------------+-------------+

--- Statistik Kedalaman Keseluruhan ---
+------------+------------+---------------+
|avg_depth_km|std_depth_km|median_depth_km|
+------------+------------+---------------+
|      124.76|      146.21|           39.1|
+------------+------------+---------------+

✓ Hasil Analisis 3 tersimpan ke HDFS


### Interpretasi Analisis 3
Gempa **dangkal (< 70 km)** umumnya paling berbahaya karena energinya tidak terdisipasi
sebelum mencapai permukaan. Indonesia, yang berada di zona subduksi, sering mengalami
gempa dangkal akibat pergerakan lempeng tektonik di kedalaman rendah.
Gempa dalam (> 300 km) meski magnitudonya besar, dampaknya di permukaan cenderung
lebih kecil karena energi teredam oleh lapisan bumi yang tebal.

In [6]:
# : Simpan ringkasan semua hasil ke spark_results.json untuk dashboard
import os

DASHBOARD_DATA_DIR = os.path.join("..", "dashboard", "data")
os.makedirs(DASHBOARD_DATA_DIR, exist_ok=True)

# Konversi ke Python dict
mag_data     = hasil_mag.toPandas().to_dict(orient="records")
wilayah_data = hasil_wilayah.toPandas().to_dict(orient="records")
depth_data   = hasil_kedalaman.toPandas().to_dict(orient="records")

spark_results = {
    "generated_at": __import__('datetime').datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "total_events": total,
    "analisis_1_distribusi_magnitudo": mag_data,
    "analisis_2_wilayah_aktif": wilayah_data,
    "analisis_3_distribusi_kedalaman": depth_data,
}

output_path = os.path.join(DASHBOARD_DATA_DIR, "spark_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2)

print(f"✓ spark_results.json tersimpan di: {output_path}")
print(f"  Total event dianalisis: {total}")
print("\n=== Semua analisis selesai! ===")

✓ spark_results.json tersimpan di: ../dashboard/data/spark_results.json
  Total event dianalisis: 30

=== Semua analisis selesai! ===


/tmp/ipykernel_11437/3817650050.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": __import__('datetime').datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),


In [7]:
# Verifikasi output lokal
import os
print("=== Isi spark/data/hasil/ ===")
for root, dirs, files in os.walk("data/hasil"):
    for file in files:
        print(os.path.join(root, file))

spark.stop()
print("✓ SparkSession ditutup")

=== Isi spark/data/hasil/ ===
data/hasil/distribusi_kedalaman/part-00000-c570396f-0de1-4c57-9e24-70ced548dbc9-c000.json
data/hasil/distribusi_kedalaman/_SUCCESS
data/hasil/distribusi_kedalaman/._SUCCESS.crc
data/hasil/distribusi_kedalaman/.part-00000-c570396f-0de1-4c57-9e24-70ced548dbc9-c000.json.crc
data/hasil/wilayah_aktif/part-00000-288b73cd-9c79-44d0-b31d-c1cf39f4e694-c000.json
data/hasil/wilayah_aktif/_SUCCESS
data/hasil/wilayah_aktif/._SUCCESS.crc
data/hasil/wilayah_aktif/.part-00000-288b73cd-9c79-44d0-b31d-c1cf39f4e694-c000.json.crc
data/hasil/distribusi_magnitudo/_SUCCESS
data/hasil/distribusi_magnitudo/._SUCCESS.crc
data/hasil/distribusi_magnitudo/.part-00000-7fa139c8-1bac-4c9c-84be-d983335c070d-c000.json.crc
data/hasil/distribusi_magnitudo/part-00000-7fa139c8-1bac-4c9c-84be-d983335c070d-c000.json
✓ SparkSession ditutup
